# Game loop: wiring vision, arm, engine and teacher together

**Deliverable:** the complete game loop, runnable without hardware. Everything is injected:

| Dependency | Real robot | Gazebo | Software (this notebook) |
|---|---|---|---|
| vision | `gambit_vision` board reader | camera topic + vision | `FakeVision` (reads the simulated board) |
| arm | `gambit_arm` primitives | `gambit_sim` teleport | `FakeArm` (edits the simulated board) |
| robot policy | teacher (`gambit_teach`) | same | `EnginePolicy(MaterialEngine())` |
| human | a person | a script | `SimulatedHuman` (any engine) |

`SimulatedGame` plays engine vs random entirely in software: the simulated human moves pieces on a
shared "physical" piece map, the fake vision reads that map, the game manager detects the move,
the robot plans arm steps, the fake arm executes them, and the camera verifies. Logs are written
with `logger.ipynb`, think times collected with `hesitation.ipynb`.

**TODO checklist**
- [x] Fakes + SimulatedGame (runs in the self-check).
- [ ] Fault injection: `FakeArm(fail_rate=...)`, `FakeVision(noise=...)` to exercise recovery once
      `recovery.choose_recovery_action` exists.
- [ ] Real wiring in `run_play_game.ipynb` (vision node / arm driver adapters).

In [ ]:
%run ./game_manager.ipynb
%run ./logger.ipynb
%run ./hesitation.ipynb

In [ ]:
import random
import tempfile
from collections import Counter
from pathlib import Path
from typing import Optional

import chess

## The simulated physical world

`PhysicalBoard` is the ground truth the fake camera sees: a `{square: Piece}` map plus a graveyard
counter. The fake arm and the simulated human both change it; the game manager only ever sees it
through `FakeVision`.

In [ ]:
class PhysicalBoard:
    def __init__(self, fen: str = chess.STARTING_FEN, spare_queens: int = SPARE_QUEENS):
        self.pieces = chess.Board(fen).piece_map()
        self.graveyard = Counter(Graveyard(spare_queens).pieces)

    def observation(self, t: float = 0.0) -> BoardObservation:
        bb = chess.BaseBoard.empty()
        for sq, p in self.pieces.items():
            bb.set_piece_at(sq, p)
        return BoardObservation(bb.board_fen(), [1.0] * 64, t)


class FakeArm:
    """ArmInterface on a PhysicalBoard. `fail_rate`: chance a primitive does nothing and returns False."""

    def __init__(self, world: PhysicalBoard, fail_rate: float = 0.0, seed: int = 0):
        self.world, self.fail_rate, self.rng = world, fail_rate, random.Random(seed)
        self.held: Optional[chess.Piece] = None
        self.calls: list[str] = []

    def _fail(self) -> bool:
        return self.rng.random() < self.fail_rate

    def pick(self, square):
        self.calls.append(f"pick {chess.square_name(square)}")
        if self._fail() or self.held or square not in self.world.pieces:
            return False
        self.held = self.world.pieces.pop(square)
        return True

    def place(self, square):
        self.calls.append(f"place {chess.square_name(square)}")
        if self._fail() or self.held is None or square in self.world.pieces:
            return False
        self.world.pieces[square], self.held = self.held, None
        return True

    def move_piece(self, from_sq, to_sq):
        return self.pick(from_sq) and self.place(to_sq)

    def capture_to_graveyard(self, square):
        if not self.pick(square):
            return False
        self.calls.append("drop graveyard")
        self.world.graveyard[self.held] += 1
        self.held = None
        return True

    def place_from_graveyard(self, piece, square):
        self.calls.append(f"graveyard {piece.symbol()} -> {chess.square_name(square)}")
        if self.world.graveyard[piece] <= 0 or self.held:
            return False
        self.world.graveyard[piece] -= 1
        self.held = piece
        return self.place(square)

    def park(self):
        return True


class FakeVision:
    """Reads the PhysicalBoard. `before_read` lets the simulated human act between frames."""

    def __init__(self, world: PhysicalBoard, clock=None):
        self.world, self.before_read = world, None
        self.clock = clock or (lambda: 0.0)

    def read(self) -> BoardObservation:
        if self.before_read:
            self.before_read()
        return self.world.observation(self.clock())


class SimulatedHuman:
    """Moves pieces on the PhysicalBoard when it is the human's turn on the manager's rules board."""

    def __init__(self, engine: EngineLike, world: PhysicalBoard, color: chess.Color):
        self.engine, self.world, self.color = engine, world, color
        self._moved_at_ply = None

    def maybe_move(self, rules_board: chess.Board) -> None:
        if rules_board.turn != self.color or rules_board.is_game_over() or self._moved_at_ply == rules_board.ply():
            return
        move = self.engine.play(rules_board)
        gy = Graveyard(); gy.pieces = Counter(self.world.graveyard)
        try:
            steps = plan_physical_move(rules_board, move, gy)
        except MissingPieceError:                       # a human just brings the piece from the box
            steps = plan_physical_move(rules_board, chess.Move(move.from_square, move.to_square, chess.QUEEN), gy)
        self.world.pieces = apply_steps(self.world.pieces, steps, self.world.graveyard)
        for s in steps:
            if s.op == FROM_GRAVEYARD:
                self.world.graveyard[s.piece] -= 1
        self._moved_at_ply = rules_board.ply()

## MoveEvent -> log record

`make_log_listener` turns game-manager events into `MoveRecord`s. If an `analyst` engine is given,
each position is analysed for the eval and best move (that is what the teaching side needs).

In [ ]:
def make_log_listener(logger: GameLogger, analyst: Optional[EngineLike] = None):
    def listener(event: MoveEvent) -> None:
        before = chess.Board(event.fen_before)
        after = before.copy(); after.push(event.move)
        eval_cp = best = None
        if analyst is not None:
            cands = analyst.analyse(before, multipv=1)
            best = cands[0].move.uci() if cands else None
            if not after.is_game_over():
                a = analyst.analyse(after, multipv=1)
                if a:
                    eval_cp = a[0].score_cp if after.turn == chess.WHITE else -a[0].score_cp
        logger.log(MoveRecord(logger.game_id, event.ply, event.by, event.fen_before, after.fen(),
                              event.move.uci(), event.san, event.think_time_s, eval_cp, best,
                              event.robot_attempts, event.plan, extra=event.extra))
    return listener

## SimulatedGame

In [ ]:
class _TickClock:
    """Deterministic clock: each call advances 0.1 s (so think times are non-zero and reproducible)."""

    def __init__(self):
        self.t = 0.0

    def __call__(self):
        self.t += 0.1
        return self.t


class SimulatedGame:
    def __init__(self, robot_policy: MovePolicy, human_engine: EngineLike,
                 human_color: chess.Color = chess.WHITE, fen: str = chess.STARTING_FEN,
                 log_dir: Optional[Path] = None, analyst: Optional[EngineLike] = None,
                 arm_fail_rate: float = 0.0):
        self.clock = _TickClock()
        self.world = PhysicalBoard(fen)
        self.arm = FakeArm(self.world, arm_fail_rate)
        self.vision = FakeVision(self.world, self.clock)
        self.human = SimulatedHuman(human_engine, self.world, human_color)
        self.hesitation = HesitationTracker()
        listeners = [self.hesitation]
        self.logger = None
        if log_dir is not None:
            self.logger = GameLogger(log_dir, headers={
                "White": "sim-human" if human_color == chess.WHITE else "Gambit",
                "Black": "Gambit" if human_color == chess.WHITE else "sim-human"}, start_fen=fen)
            listeners.append(make_log_listener(self.logger, analyst))
        self.manager = GameManager(self.vision, self.arm, robot_policy, human_color,
                                   board=chess.Board(fen), listeners=listeners, clock=self.clock)
        self.vision.before_read = self._human_turn

    def _human_turn(self) -> None:
        if self.manager.state is State.WAIT_HUMAN:      # the human only moves while we wait for them
            self.human.maybe_move(self.manager.board)

    def play(self, max_plies: Optional[int] = None) -> chess.Board:
        board = self.manager.run(max_plies=max_plies)
        if self.logger:
            self.logger.finish(board.result(claim_draw=True) if board.is_game_over(claim_draw=True) else "*")
        return board

## Self-check: a few plies of MaterialEngine (robot) vs RandomEngine (human), all in software

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    sim = SimulatedGame(EnginePolicy(MaterialEngine(depth=1)), RandomEngine(seed=3), chess.WHITE,
                        log_dir=Path(tmp), analyst=MaterialEngine(depth=1))
    board = sim.play(max_plies=8)
    assert board.ply() == 8, board.ply()
    assert sim.world.observation().placement == board.board_fen()          # physical == rules board
    assert State.VERIFY in sim.manager.trace and State.RECOVER not in sim.manager.trace
    df = read_jsonl(sim.logger.jsonl_path)
    assert list(df.by) == ["human", "robot"] * 4 and df.eval_cp.notna().all()
    g, = read_pgn_games(sim.logger.pgn_path)
    assert list(g.mainline_moves()) == board.move_stack
    assert len(sim.hesitation.features()) == 4

# robot promotes using a spare queen from the graveyard (robot is white, human is black)
sim = SimulatedGame(EnginePolicy(MaterialEngine(depth=1)), RandomEngine(seed=0), chess.BLACK,
                    fen="8/P6k/8/8/8/8/8/K7 w - - 0 1")
board = sim.play(max_plies=1)
assert board.move_stack[0].promotion == chess.QUEEN, board.move_stack
assert sim.world.graveyard[chess.Piece(chess.QUEEN, chess.WHITE)] == SPARE_QUEENS - 1
assert sim.world.graveyard[chess.Piece(chess.PAWN, chess.WHITE)] == 1
assert any("graveyard Q -> a8" in c for c in sim.arm.calls)
print("game_loop self-check passed")